In [1]:
import xgboost, dask, distributed, sys

print("xgboost:", xgboost.__version__)
print("dask:", dask.__version__)
print("distributed:", distributed.__version__)
print("python:", sys.version)

xgboost: 2.1.4
dask: 2024.12.1
distributed: 2024.12.1
python: 3.11.12 (main, Apr  9 2025, 08:55:55) [GCC 13.3.0]


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from newutils.data.datasets import load_train_subset, add_lags
from newutils.other.helpers import init_context
from oldutils.types import TimeRange

init_context()

In [4]:
HORIZON_HISTORY = TimeRange.YEAR
HORIZON_FORECAST = TimeRange.WEEK

COL_GAUGE_ID = "gauge_id"
TARGETS = ["q_mm_day"]

In [5]:
df = load_train_subset()

In [6]:
lags = range(1, HORIZON_HISTORY + 1)
lags_columns = ["prcp", "t_max", "t_min", "t_mean", "q_mm_day", "lvl_sm"]
df = add_lags(df, lags_columns, lags)

In [7]:
targets = []
df = add_lags(df, targets, range(1, HORIZON_FORECAST + 1), targets)
targets.extend(TARGETS)

In [8]:
df = df.dropna()

In [9]:
X = df.drop(columns=set(lags_columns + targets))
y = df[targets]
X_train, y_train = X, y

# Model

In [10]:
from dask.distributed import Client, LocalCluster, wait
import dask

# dask.config.set(scheduler="processes")

# from dask_cuda import LocalCUDACluster
import gc

# try:
#     client.close()
#     cluster.close()
#     print("Delete")
# except NameError:
#     print("Nothing")
#     pass
# gc.collect()

# cluster = LocalCluster(dashboard_address=":8789", asynchronous=True)
# # cluster = LocalCUDACluster(CUDA_VISIBLE_DEVICES="0", n_workers=1, dashboard_address=":8789")
# client = cluster.get_client()
# client = Client()

# print(cluster)
# print(client)

In [14]:
import dask.array as da
import pandas as pd
import numpy as np
import dask.dataframe as dd
import xgboost.dask as dxgb

# n_rows = 10000
# pdf = pd.DataFrame({"value": np.random.randn(n_rows)})

# df = dd.from_pandas(pdf, npartitions=10)
# print(f"Number of partitions: {df.npartitions}")


# def add_shifts(pdf):
#     pdf["shift1"] = pdf["value"].shift(1)
#     pdf["shift2"] = pdf["value"].shift(2)
#     return pdf


# df = df.map_partitions(add_shifts)
# df = df.dropna()

# X = df[["shift1", "shift2"]]
# y = df["value"]

# dtrain = dxgb.DaskDMatrix(client, X_train, y_train)

params = {
    "objective": "reg:squarederror",
    "max_depth": 3,
    "eta": 0.1,
    "tree_method": "hist",
}


# async def main():
def main():

    # Clean up any previous cluster/client
    try:
        client.close()
        cluster.close()
        print("Deleted previous client and cluster")
    except NameError:
        print("Nothing to delete")
    gc.collect()

    # 3. Start cluster & client in async mode
    # async 
    with LocalCluster(dashboard_address=":8789", asynchronous=False) as cluster:
        # async
        with Client(cluster, asynchronous=False) as client:
            print("Dask client running in async mode:", client)

            # 5. Build the DaskDMatrix and train
            # X_trainp, y_trainp = await client.persist([X_train, y_train])
            X_trainp, y_trainp = client.persist([X_train, y_train])
            wait([X_trainp, y_trainp])
            dtrain = dxgb.DaskDMatrix(client, X_trainp, y_trainp)
            # booster, training_history = await dxgb.train(
            booster, training_history = dxgb.train(
                client, params, dtrain, num_boost_round=10
            )

            print("Training metrics and booster information:")
            print(training_history)
            print(booster)


result = main()
print(result)

# params["device"] = "cuda"


# def build_dmatrix(client, X, y):
#     return dxgb.DaskDMatrix(client, X, y)

# dtrain = client.sync(build_dmatrix, client, X_train, y_train)
# output = client.sync(dxgb.train, client, params, dtrain, num_boost_round=10)

# output = dxgb.train(client, params=params, dtrain=dtrain, num_boost_round=10)

# print("Training metrics and booster information:")
# print(output)

Nothing to delete
Dask client running in async mode: <Client: 'tcp://127.0.0.1:33471' processes=4 threads=4, memory=7.66 GiB>


/home/khuzin/Projects/2025-Project-188/code/src/newutils/data/datasets.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pdf[_gen_lag_name(column, lag)] = pdf[column].shift(lag)
/home/khuzin/Projects/2025-Project-188/code/src/newutils/data/datasets.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pdf[_gen_lag_name(column, lag)] = pdf[column].shift(lag)
/home/khuzin/Projects/2025-Project-188/code/src/newutils/data/datasets.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fr

KeyError: 'date32[day][pyarrow]'

In [ ]:
import gc

client.close()
cluster.close()
gc.collect()

0